# Random-Walk Hypergraph Embeddings: Experiment

This notebook:
1. generates synthetic hypergraph datasets,
2. learns node embeddings with three methods,
3. evaluates them on node classification (Micro-F1, Macro-F1),
4. compares average performance.


In [3]:
import warnings
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.synthetic import generate_dataset
from src.utils import (
    clique_node2vec_embeddings,
    hyper2vec_embeddings,
    hmrw_embeddings,
)

warnings.filterwarnings("default")
np.random.seed(42)

In [4]:
def evaluate_embeddings(X, y, seed=42):
    """5-fold node-classification eval using logistic regression."""
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    clf = LogisticRegression(max_iter=2000)
    scores = cross_validate(
        clf,
        X,
        y,
        cv=cv,
        scoring={"micro_f1": "f1_micro", "macro_f1": "f1_macro"},
        n_jobs=-1,
        return_train_score=False,
    )
    return {
        "micro_f1": float(np.mean(scores["test_micro_f1"])),
        "macro_f1": float(np.mean(scores["test_macro_f1"])),
    }


In [ ]:
# --- experiment setup ---
num_datasets = 5
base_seed = 100

data_params = dict(
    n=500,
    avg_degree=10,
    num_communities=3,
    intra_strength=0.85,
    avg_edge_size=3,
    max_hyperedge_size=8,
)

embed_params = dict(
    num_walks=300,
    walk_length=40,
    window=5,
    dim=64,
    workers=1,
    p=2.0,
    q=2.0,
)

hmrw_params = dict(
    p_edge=2.0,
    q_edge=2.0,
)

method_fns = {
    "clique_node2vec": clique_node2vec_embeddings,
    "hyper2vec": hyper2vec_embeddings,
    "hmrw": hmrw_embeddings,
}

print("Configured methods:", list(method_fns.keys()))
print("Bias params: p=2, q=2, p_edge=2, q_edge=2")

Configured methods: ['clique_node2vec', 'hyper2vec', 'hmrw']
Bias params: p=2, q=2, p_edge=2, q_edge=2


In [19]:
rows = []

for ds_i in range(num_datasets):
    seed = base_seed + ds_i
    sample = generate_dataset(seed=seed, **data_params)
    H = sample["H"]

    # Align labels with the actual nodes present in H (some generators may drop isolated nodes).
    nodes = sorted(H.get_nodes())
    y = np.array([sample["g1"][u] for u in nodes])

    print(f"Dataset {ds_i+1}/{num_datasets} (seed={seed}, n_nodes={len(nodes)})")

    for name, fn in method_fns.items():
        if name == "hmrw":
            X = fn(H, seed=seed, **embed_params, **hmrw_params)
        else:
            X = fn(H, seed=seed, **embed_params)
        scores = evaluate_embeddings(X, y, seed=seed)
        rows.append({"dataset": ds_i, "method": name, **scores})
        print(f"  {name:>15} | micro={scores['micro_f1']:.4f} macro={scores['macro_f1']:.4f}")

results = pd.DataFrame(rows)
results.head()

generate_dataset params -> n=1000, avg_degree=10, num_communities=3, intra_strength=0.85, avg_edge_size=3, max_hyperedge_size=8, seed=100
Dataset 1/5 (seed=100, n_nodes=944)
  clique_node2vec | micro=0.3326 macro=0.2862


KeyboardInterrupt: 

In [17]:
results.sort_values("micro_f1", ascending=False)

,dataset,method,micro_f1,macro_f1
8,2,hmrw,0.411336,0.376279
5,1,hmrw,0.392982,0.376587
2,0,hmrw,0.385425,0.317212
7,2,hyper2vec,0.385155,0.314139
4,1,hyper2vec,0.376518,0.356138
1,0,hyper2vec,0.375169,0.287484
10,3,hyper2vec,0.371835,0.272800
9,3,clique_node2vec,0.361735,0.180635
6,2,clique_node2vec,0.359379,0.177579
3,1,clique_node2vec,0.345614,0.211103


In [15]:
summary = (
    results.groupby("method", as_index=False)[["micro_f1", "macro_f1"]]
    .agg(["mean", "std"])
)
summary

method  micro_f1            macro_f1          
                        mean       std      mean       std
0  clique_node2vec  0.338015  0.024992  0.198387  0.019449
1             hmrw  0.355388  0.067795  0.316565  0.062686
2        hyper2vec  0.368051  0.020973  0.309457  0.031939

In [ ]:
import matplotlib.pyplot as plt

plot_df = results.groupby("method", as_index=False)[["micro_f1", "macro_f1"]].mean()
ax = plot_df.plot(x="method", y=["micro_f1", "macro_f1"], kind="bar", figsize=(8, 4))
ax.set_ylabel("F1 score")
ax.set_title("Average performance across synthetic datasets")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()